# SentinelDesk triage: train on Colab (from VS Code)

**How to connect (VS Code + Google Colab extension):** open this notebook → **Select Kernel** (top right) → **Colab** → **New Colab Server** → pick a **T4 GPU** runtime. Then run the cells top to bottom.

The Colab runtime is a remote machine: it does **not** see your laptop files. So we `git clone` the repo (push your branch first) and send the trained artifacts back at the end.

Rules that still apply here (see `CLAUDE.md`): the split comes only from `split.py`; nothing reads the holdout labels except `eval_triage.py`, and that runs **once, at the very end**.

In [ ]:
import platform, os
print(platform.system(), platform.node(), os.getcwd())

In [23]:
# 1. Check the GPU (expect a Tesla T4). If this fails, switch the runtime to a GPU one.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Get the code
Set `BRANCH` to the branch you pushed. The repo must be reachable from Colab (public, or use a token URL).

In [24]:
REPO_URL = "https://github.com/wuhong4088/HackathonGDG.git"
BRANCH = "shatru"
WORKDIR = "/content/HackathonGDG"

import os
if os.path.exists(WORKDIR):
    !git -C {WORKDIR} fetch origin {BRANCH} && git -C {WORKDIR} checkout {BRANCH} && git -C {WORKDIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 130 (delta 68), reused 102 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 55.48 KiB | 916.00 KiB/s, done.
Resolving deltas: 100% (68/68), completed with 17 local objects.
From https://github.com/wuhong4088/HackathonGDG
 * branch            shatru     -> FETCH_HEAD
   1eb3372..377fc3c  shatru     -> origin/shatru
Already on 'shatru'
Your branch is behind 'origin/shatru' by 31 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 377fc3c Merge main into shatru: keep data-audit/RAG rows and triage Done row in status board
From https://github.com/wuhong4088/HackathonGDG
 * branch            shatru     -> FETCH_HEAD
   1eb3372..377fc3c  shatru     -> origin/shatru
Already on 'shatru'
Your branch is behind 'origin/shatru' by 31 commits, and can be fast-forwarded.
  (use "git pu

In [ ]:
# 3. Install dependencies (keeps Colab's CUDA torch). scikit-learn must be 1.7.2 so the laptop can load the models.
import os, sys, subprocess
req = f"{WORKDIR}/requirements.txt"
if not os.path.exists(req):
    raise SystemExit(f"{req} not found: cell 2 (git clone) did not finish. Re-run cell 2 and check it prints the commit.")
if "sklearn" in sys.modules:
    print("WARNING: sklearn was already imported in this kernel -> after install, Restart and re-run from cell 2.")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-4000:])
if r.returncode != 0:
    raise SystemExit("pip install failed: copy the red text above and send it over.")
import sklearn, numpy, torch
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
if sklearn.__version__ != "1.7.2":
    raise SystemExit("Old scikit-learn still loaded: use Restart (notebook toolbar), then run cell 2 and this cell again.")

## 4. Settings
* `EMBED_MODEL`: **`intfloat/multilingual-e5-base`** is the team decision (README "Models (decided)"). Only with this model does step 6 also build the **shared index** `data/index/emb.npy` + `kb.pkl` that retrieval uses. `e5-small` is only for quick experiments.
* Triage artifacts go to `/content/artifacts`; the shared index goes to `data/index/` inside the repo (README spec). Step 9 exports both.

In [25]:
EMBED_MODEL = "intfloat/multilingual-e5-base"   # team model: do not change unless the team re-decides
ARTIFACTS = "/content/artifacts"
INDEX_DIR = f"{WORKDIR}/data/index"
os.environ["SENTINELDESK_ARTIFACTS"] = ARTIFACTS
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Optional: an HF token avoids Hub rate limits (Colab secrets don't work from VS Code, so paste it here).
# from getpass import getpass; os.environ["HF_TOKEN"] = getpass("HF token (optional): ")

In [26]:
# 5. Tests + a quick smoke run on 2,000 tickets (~1-2 min). Catches problems before the full run.
!python -m pytest -q tests/test_triage.py
!python scripts/build_split.py --sample 2000 && python scripts/embed.py --sample 2000 --model {EMBED_MODEL} && python scripts/train_triage.py --sample 2000 --model {EMBED_MODEL}

...........s......                                                       [100%]
17 passed, 1 skipped in 2.08s
19:10:37 INFO sentineldesk.triage.data: kb=22869 test=5718  missing subjects in kb=3102
19:10:37 INFO sentineldesk.triage.data: type: ['Change', 'Incident', 'Problem', 'Request']
19:10:37 INFO sentineldesk.triage.data: queue: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']
19:10:37 INFO sentineldesk.triage.data: priority: ['high', 'low', 'medium']
19:10:37 INFO sentineldesk.triage.data: wrote splits to /content/artifacts/sample_2000/splits: {'train': 2000, 'half_a': 111, 'half_b': 111, 'test': 556}
19:10:51 INFO datasets: TensorFlow version 2.21.0 available.
19:10:51 INFO datasets: JAX version 0.11.1 available.
19:10:53 INFO httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolv

## 6. Full training
Expected on a T4: split <1 min, embeddings ~3-5 min (e5-base; first run also downloads ~1 GB), training ~4 min.
The embed step embeds the 22,869 kb tickets **once**, writes the shared index to `data/index/` and slices triage's train / half A / half B from it. Run the three cells **in order**: training needs the embeddings of the same model.

In [27]:
%%time
!python scripts/build_split.py

19:11:20 INFO sentineldesk.triage.data: kb=22869 test=5718  missing subjects in kb=3102
19:11:20 INFO sentineldesk.triage.data: type: ['Change', 'Incident', 'Problem', 'Request']
19:11:20 INFO sentineldesk.triage.data: queue: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']
19:11:20 INFO sentineldesk.triage.data: priority: ['high', 'low', 'medium']
19:11:20 INFO sentineldesk.triage.data: wrote splits to /content/artifacts/splits: {'train': 20582, 'half_a': 1144, 'half_b': 1143, 'test': 5718}
CPU times: user 6.56 ms, sys: 1.95 ms, total: 8.51 ms
Wall time: 1.51 s


In [28]:
%%time
# Embeds kb once -> data/index/emb.npy (22869, 768) + kb.pkl, and triage parts; then the team's leakage check.
!python scripts/embed.py --model {EMBED_MODEL}
!python check_rules.py
!cat {INDEX_DIR}/meta.json

19:11:30 INFO datasets: TensorFlow version 2.21.0 available.
19:11:30 INFO datasets: JAX version 0.11.1 available.
19:11:32 INFO httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
19:11:32 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
19:11:32 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json?%2Fintfloat%2Fmultilingual-e5-base%2Fresolve%2Fmain%2Fmodules.json=&etag=%22ac2039abdf6ff023b27c919bb9675cfe378cb10f%22 "HTTP/1.1 200 OK"
19:11:32 INFO httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
19:11:32 INFO sentence_transformers.base.model: Loading SentenceTransform

In [29]:
%%time
# Trains 0, 1, 5 on train; ensemble on half A; temperature + thresholds on half B. Prints half-B metrics: accuracy + Macro-F1 per task (fallback vs full), critical recall/precision, suggested thresholds.
!python scripts/train_triage.py --model {EMBED_MODEL}

19:12:29 INFO sentineldesk.triage.training: ▶ model 0 (embeddings + LR) ...
19:13:11 INFO sentineldesk.triage.training: ✔ model 0 (embeddings + LR): 41.5s
19:13:11 INFO sentineldesk.triage.training: ▶ critical detector (5) ...
19:13:23 INFO sentineldesk.triage.training: ✔ critical detector (5): 12.3s
19:13:23 INFO sentineldesk.triage.training: critical: {'threshold': 0.4429355170369817, 'target_recall': 0.9, 'half_b_recall': 0.9045226130653267, 'half_b_precision': 0.3474903474903475, 'half_b_positives': 199}
19:13:23 INFO sentineldesk.triage.training: ▶ model 1 (TF-IDF + LR) ...
19:16:48 INFO sentineldesk.triage.training: ✔ model 1 (TF-IDF + LR): 205.6s
19:16:48 INFO sentineldesk.triage.training: ▶ model 2 (neighbour index + votes) ...
19:16:48 INFO faiss.loader: Loading faiss with AVX512 support.
19:16:48 INFO faiss.loader: Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
19:16:48 INFO faiss.loader: Loading faiss with A

In [30]:
# 7. Sanity check: load the trained Triage and predict two live tickets (English + German, one without subject).
import sys, json; sys.path.insert(0, WORKDIR)
from sentineldesk.triage.predict import Triage
triage = Triage.load(ARTIFACTS)
print("model:", triage.meta["embedding_model"], "| mode:", triage.mode)
for subj, body in [("VPN outage", "Since 8am nobody in the clinic can reach the EMR system, all users affected."),
                   (None, "Guten Tag, ich habe eine Frage zu meiner letzten Rechnung. Können Sie mir eine Kopie schicken?")]:
    r = triage.predict(subj, body)
    print(json.dumps({k: r[k] for k in ("mode", "critical", "latency_ms")} |
                     {t: (r[t]["label"], r[t]["confidence"]) for t in ("type", "queue", "priority")}, ensure_ascii=False, indent=1))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model: intfloat/multilingual-e5-base | mode: full
{
 "mode": "full",
 "critical": {
  "score": 0.8411,
  "threshold": 0.4429,
  "fires": true,
  "keyword_hits": [
   "all users",
   "emr",
   "outage"
  ]
 },
 "latency_ms": {
  "embed": 390.26,
  "models": 3.19,
  "neighbours": 6.13,
  "tfidf": 1.88,
  "ensemble": 0.68,
  "critical": 0.34,
  "total": 402.57
 },
 "type": [
  "Incident",
  0.9033
 ],
 "queue": [
  "Technical Support",
  0.3934
 ],
 "priority": [
  "high",
  0.5146
 ]
}
{
 "mode": "full",
 "critical": {
  "score": 0.0418,
  "threshold": 0.4429,
  "fires": false,
  "keyword_hits": []
 },
 "latency_ms": {
  "embed": 11.79,
  "models": 2.54,
  "neighbours": 5.96,
  "tfidf": 1.57,
  "ensemble": 0.58,
  "critical": 0.4,
  "total": 22.9
 },
 "type": [
  "Request",
  0.5045
 ],
 "queue": [
  "Billing and Payments",
  0.9547
 ],
 "priority": [
  "medium",
  0.3912
 ]
}


## 8. Holdout evaluation (ONCE, at the very end)
Only run this when the models are final. Never change keywords, thresholds or hyperparameters after looking at these numbers.

In [ ]:
RUN_HOLDOUT_EVAL = False   # set True only for the final run
if RUN_HOLDOUT_EVAL:
    !python scripts/eval_triage.py
    !mkdir -p {ARTIFACTS}/results && cp -r results/triage_metrics.json results/plots {ARTIFACTS}/results/

## 9. Send the artifacts back to your laptop
The VS Code Colab extension does not support `drive.mount()` or `files.download()` reliably, so try these in order:

1. **Google Drive** (works if mounting succeeds in your extension version)
2. **Private Hugging Face repo** (most reliable; needs an HF *write* token)
3. Fall back to the Colab file browser / "Download" on `/content/sentineldesk_artifacts.zip`

The zip holds `artifacts/` (`triage/` = all the demo needs, plus `embeddings/` and `splits/`) and `index/` (the team's shared `emb.npy` + `kb.pkl` + `meta.json` for retrieval; share it through Drive as the README says).

In [ ]:
# Build the zip: triage artifacts (skips quick-run sample folders) + the shared index under index/.
!rm -rf /content/export && mkdir -p /content/export && cp -r {ARTIFACTS} /content/export/artifacts && rm -rf /content/export/artifacts/sample_* && cp -r {INDEX_DIR} /content/export/index
!cd /content/export && rm -f /content/sentineldesk_artifacts.zip && zip -qr /content/sentineldesk_artifacts.zip . && ls -lh /content/sentineldesk_artifacts.zip

In [ ]:
# Option 1: Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/sentineldesk && cp /content/sentineldesk_artifacts.zip /content/drive/MyDrive/sentineldesk/
    print("Copied to My Drive/sentineldesk/sentineldesk_artifacts.zip")
except Exception as e:
    print("Drive mount not available here:", e, "-> use option 2")

In [ ]:
# Option 2: private Hugging Face model repo (artifacts derive from a CC-BY-NC dataset: keep it private).
from getpass import getpass
from huggingface_hub import HfApi
HF_REPO = "YOUR_HF_USERNAME/sentineldesk-artifacts"   # <- change
api = HfApi(token=getpass("HF write token: "))
api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_file(path_or_fileobj="/content/sentineldesk_artifacts.zip", path_in_repo="sentineldesk_artifacts.zip",
                repo_id=HF_REPO, repo_type="model")
print(f"Uploaded. On the laptop: hf download {HF_REPO} sentineldesk_artifacts.zip --local-dir .")

## On the laptop
```bash
# from the repo root, with the zip downloaded next to it
unzip -o sentineldesk_artifacts.zip -d /tmp/sd_export
mkdir -p artifacts data/index
cp -r /tmp/sd_export/artifacts/. artifacts/ && cp -r /tmp/sd_export/index/. data/index/
python check_rules.py     # must print OK (also checks data/index/kb.pkl)
python -c "from sentineldesk.triage.predict import Triage; print(Triage.load('artifacts').predict(None, 'VPN down for all users')['critical'])"
```